In [ ]:
# ===== Cell 1 of 4 - set up =====
# Run this first, and again whenever Python restarts. It makes the three widgets - 01 LLM endpoint,
# 02 LLM token, and 03 Project Name (can be a model ID), which names the project's folder in Projects -
# and takes your user id from Databricks. It installs from PyPI only what the engine lacks; if it had to, it restarts Python and asks you
# to run this cell once more. It then gets flowR ready, the program that reads the model's R code, and says
# what to do next.
import os, sys; sys.path.insert(0, os.path.abspath("engine")); import verifier
verifier.setup(dbutils)


In [ ]:
# ===== Cell 2 of 4 - your chat(), and where your files go =====
# Paste your organisation's chat() below, in place of this one. chat(SystemPrompt, MainPrompt) must return a
# dictionary holding the model's reply under "answer". It must read the endpoint, the token and your user id
# inside the function, when it is called - verifier.live("llm_endpoint"), verifier.live("llm_token") and
# verifier.live("reviewer_id") - so that a fresh token pasted into widget 02 is used by the next call.
# Cell 3 calls it from many threads at once, up to 256, so it must keep nothing from one call to the next.
# Each question and its answer stay within 40,000 tokens. This cell asks chat() one short question to check
# it answers, then makes the project's three input folders and says what goes in each.
import requests

def chat(SystemPrompt, MainPrompt, history=[]):
    payload = {
        "app": "sparkair",
        "enable_streaming": False,
        "flow_name": "general_chat",
        "history": history,
        "optionalParameter": {
            "maxtoken": 250000,
            "contextlength": 250000,
            "Temperature": 0.01,           # low: the same question gives the same answer
            "Top_k": 1,
            "Penalty": 1.1,
            "DefaultPrompt": SystemPrompt,
        },
        "query": MainPrompt,
        "select_all": False,
    }
    headers = {
        "Authorization": f'Bearer {verifier.live("llm_token")}',
        "SP_SSO_UID": verifier.live("reviewer_id"),
        "Content-Type": "application/json",
    }
    response = requests.post(verifier.live("llm_endpoint"), json=payload, headers=headers, timeout=180)
    response.raise_for_status()
    return response.json()          # the engine reads the model's reply from "answer"

verifier.check_chat(chat)           # asks it one question, then makes the input folders and says what goes where


In [ ]:
# ===== Cell 3 of 4 - read the inputs and run the review =====
# Reads every input file into numbered units and links the pieces of the model's package with flowR. Then it
# asks your chat() to explain each piece of the model's code, and to find the chunks of the methodology behind
# each piece and where the code may depart from them - many questions at once, with a line of progress every
# minute. It writes into the project folder, beside the three input folders: Output.xlsm, and the _Audit folder, the
# record of the run: Audit_Log.xlsx and a numbered folder of JSON files for each step. If an input folder is empty, it says which and stops. Run it again when it says a step
# did not finish - it says what chat() last returned: if the token ran out, paste a fresh one into widget 02
# first; if the gateway was down (a 503, say), wait until it is back. Running every cell again, from cell 1,
# is as good: a finished step is never repeated, and no answer already received is asked for again, even if
# you interrupted the cell. After Python restarts, run cells 1 and 2 first: cell 3 then carries the project's
# run on from its audit log - only answers an interrupted cell still held in memory are asked for again. If an
# file, the engine or a setting has changed, it starts a new run instead and replaces both files - but never
# an Output.xlsm you have edited: move that one aside first.
verifier.review()


In [ ]:
# ===== Cell 4 of 4 - check Output.xlsm and the _Audit folder against their own record =====
# Run it once cell 3 has finished. Nine checks, each Confirmed or Not confirmed: the input files are the ones
# fingerprinted; the engine's files are the ones that produced the run; reading the methodology, the
# documentation and the package again gives the same units; every unit read is in the record; Output.xlsm
# carries this run's identity; no access token was written into Output.xlsm or the _Audit folder, inside
# a workbook included; and the _Audit folder holds every file its inventory lists, unchanged.
verifier.verify()
